# PS1 - event-definition study

**Question.** PS1's contract pair (Annexure 1 Table 13: precision >= 0.90 **and** recall >= 0.80)
is reachable only above a certain ROC-AUC, and that threshold depends entirely on the base rate of
the event being predicted. We measure AUC ~0.888 on GATE and ~0.894 on TVM today.

So the open question is not *which model* but *which EVENT*. This notebook measures, for every
candidate OOS event definition, the base rate at each contracted horizon (24-hour, 3-day, 7-day,
30-day) and therefore the AUC the contract would demand.

**Read-only.** Reads the PS3 episode fact from S3. No Aurora, no deploy, no Spark. Runs in
seconds on ~51k rows.

*Base rates are computed as UPPER bounds - overlapping pre-failure windows are not de-duplicated.
That biases toward optimism, so anything marked unreachable here is genuinely unreachable.*

In [ ]:
# CELL 1 : imports and configuration
import pandas as pd, numpy as np, s3fs
from math import sqrt
from statistics import NormalDist

BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
PREFIX = "chicago/ps3_outputs/ps3_device_episode_fact"

# PS1's eligible device population, from PS2's independent label recount on the
# 29-Aug data. These are the denominators that matter: the episode table only
# contains devices that had at least one episode.
ELIGIBLE_DEVICES = {"GATE": 821, "TVM": 468, "VALIDATOR": 3028}

# ROC-AUC measured on the 17-Sep PS1 runs (test split).
MEASURED_AUC = {"GATE": 0.888, "TVM": 0.894, "VALIDATOR": 0.997}

HORIZONS = [1, 3, 7, 30]   # 24-hour, 7-day and 30-day are contracted; 3-day is what exists
N = NormalDist()
print("config loaded")

In [ ]:
# CELL 2 : the AUC a base rate demands
def auc_required(base_rate, precision=0.90, recall=0.80):
    """Minimum ROC-AUC needed to hit the contract pair at this base rate.

    For precision P at recall R the largest tolerable false-positive rate is
        FPR = R*p*(1-P) / (P*(1-p))
    Under a binormal ROC, d_prime = z(TPR) - z(FPR) and AUC = Phi(d_prime/sqrt(2)).
    """
    if base_rate <= 0 or base_rate >= 1:
        return np.nan, np.nan
    fpr = recall * base_rate * (1 - precision) / (precision * (1 - base_rate))
    if fpr >= 1:
        return fpr, 0.5
    d = N.inv_cdf(recall) - N.inv_cdf(fpr)
    return fpr, N.cdf(d / sqrt(2))


# sanity check against the values quoted in the assessment document
for p, exp in ((0.039, 0.994), (0.078, 0.990), (0.157, 0.982), (0.624, 0.909)):
    f, a = auc_required(p)
    assert abs(a - exp) < 0.002, (p, a, exp)
    print(f"  p={p:.3f} -> max FPR {f:.4f}, AUC required {a:.3f}")
print("maths OK")

In [ ]:
# CELL 3 : load the newest PS3 episode run from S3
fs = s3fs.S3FileSystem()
parts = sorted(fs.glob(f"{BUCKET}/{PREFIX}/computed_date=*/run_id=*/*.parquet"))
if not parts:
    raise SystemExit(f"no parquet under s3://{BUCKET}/{PREFIX}/")
newest_date = sorted({p.split("computed_date=")[1].split("/")[0] for p in parts})[-1]
parts = [p for p in parts if f"computed_date={newest_date}" in p]
print(f"reading {len(parts)} part(s), computed_date={newest_date}")

# Read through a file handle, not a path. Passing an s3:// path lets pyarrow
# discover computed_date= / run_id= as hive partition columns, and run_id is
# ALSO a real string column inside the file -- the two cannot merge
# ("Field run_id has incompatible types: string vs dictionary").
frames = []
for p in parts:
    with fs.open(p, "rb") as fh:
        frames.append(pd.read_parquet(fh))
df = pd.concat(frames, ignore_index=True)

df["episode_start"] = pd.to_datetime(df["episode_start"])
win_start, win_end = df["episode_start"].min(), df["episode_start"].max()
win_days = max(1, (win_end - win_start).days + 1)
print(f"episodes={len(df):,}  window {win_start.date()} -> {win_end.date()}  ({win_days} days)")
print("fleets:", dict(df["mars_device_category"].value_counts()))

In [ ]:
# CELL 4 : candidate event definitions
# duration: prefer the union measure, fall back to the raw signal span
dur = df.get("oos_minutes_union")
if dur is None or dur.isna().all():
    dur = df.get("set_signal_span_minutes")
df["_dur"] = pd.to_numeric(dur, errors="coerce").fillna(0.0)
df["_svc"] = df.get("requires_service_call", pd.Series(False, index=df.index)).fillna(False).astype(bool)
df["_cnt"] = pd.to_numeric(df.get("oos_set_event_count", 0), errors="coerce").fillna(0)

DEFINITIONS = {
    "D0  every OOS Set episode (today)": lambda d: d.index == d.index,
    "D1  requires_service_call":         lambda d: d["_svc"],
    "D2  duration >= 60 min":            lambda d: d["_dur"] >= 60,
    "D3  duration >= 240 min":           lambda d: d["_dur"] >= 240,
    "D4  duration >= 1440 min (24h)":    lambda d: d["_dur"] >= 1440,
    "D5  >= 3 OOS Set events":           lambda d: d["_cnt"] >= 3,
    "D6  service call AND >= 60 min":    lambda d: d["_svc"] & (d["_dur"] >= 60),
}
print(f"duration column in use: non-zero {int(df['_dur'].gt(0).sum()):,} of {len(df):,}")
print(f"requires_service_call true: {int(df['_svc'].sum()):,}")

In [ ]:
# CELL 5 : compute base rate and required AUC for every definition x horizon
rows = []
for fleet, g in df.groupby("mars_device_category"):
    n_dev = ELIGIBLE_DEVICES.get(fleet)
    if not n_dev:
        continue
    device_days = n_dev * win_days
    for name, pred in DEFINITIONS.items():
        sub = g[pred(g)]
        eps = len(sub)
        for h in HORIZONS:
            # an onset makes the h days before it positive; overlap is not removed,
            # so this is an UPPER bound on the base rate
            p = min(0.999, eps * h / device_days) if device_days else np.nan
            fpr, auc = auc_required(p)
            rows.append(dict(fleet=fleet, definition=name, episodes=eps,
                             devices_with_eps=sub["device_id"].nunique(),
                             horizon_days=h, base_rate=p,
                             max_fpr=fpr, auc_required=auc))

out = pd.DataFrame(rows)
out["auc_today"] = out["fleet"].map(MEASURED_AUC)
out["reachable"] = np.where(out["auc_required"] <= out["auc_today"], "YES", "no")
print(f"{len(out)} fleet x definition x horizon combinations")

In [ ]:
# CELL 6 : results
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 400)

for fleet in sorted(out["fleet"].unique()):
    f = out[out["fleet"] == fleet]
    print("=" * 104)
    print(f"{fleet}   eligible devices={ELIGIBLE_DEVICES[fleet]:,}   "
          f"device-days={ELIGIBLE_DEVICES[fleet]*win_days:,}   measured AUC={MEASURED_AUC[fleet]}")
    print("=" * 104)
    print(f.pivot_table(index=["definition", "episodes"], columns="horizon_days",
                        values=["base_rate", "auc_required"], aggfunc="first")
           .round(4).to_string())
    win = f[f["reachable"] == "YES"]
    if len(win):
        print()
        print("  REACHABLE at today's AUC:")
        for _, r in win.iterrows():
            print(f"    {r['definition']:34s} horizon {r['horizon_days']:>2}d  "
                  f"base rate {r['base_rate']:.4f}  needs AUC {r['auc_required']:.3f}")
    else:
        print()
        print("  Nothing reachable at today's AUC on any definition or horizon.")
    print()

out.to_csv("ps1_event_definition_study.csv", index=False)
print("saved -> ps1_event_definition_study.csv")

In [ ]:
# CELL 7 : how close is the closest option?
# If nothing is reachable, the useful output is the smallest gap -- that is the
# number the threshold conversation in SoW section 3 should be built on.
out["auc_gap"] = out["auc_required"] - out["auc_today"]
closest = (out.sort_values("auc_gap")
              .groupby("fleet")
              .head(3)[["fleet", "definition", "horizon_days", "episodes",
                        "base_rate", "auc_required", "auc_today", "auc_gap"]])
print("Closest three combinations per fleet (smallest AUC shortfall):")
print(closest.round(4).to_string(index=False))